# 04 - YouTube Data Collection Workflow

This notebook documents the **YouTube Data API v3** collection pipeline used to gather
channel metadata and video data for the interest-heterogeneity analysis.

### Important

This notebook is **documentation-focused**. The collection cells will not execute
without valid YouTube API credentials configured in the `.env` file. See the
"Prerequisites" section below for setup instructions.

### Data Collection Overview

1. Read a list of YouTube channel IDs from the subscription network (`YoutubeNodes.csv`).
2. For each channel, use the API to fetch:
   - Channel statistics (subscriber count, video count).
   - Up to 200 most recent video titles, durations, and publication dates.
3. Save per-channel video data as CSV files for the TF-IDF interest vector pipeline.
4. Support checkpointing to resume interrupted collection runs.

### Quota Management

The YouTube Data API enforces a daily quota of 10,000 units. Each API call costs 1 unit.
The collector tracks cumulative quota usage and stops when the configured limit is reached.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import pandas as pd

from src.collection.youtube_collector import YouTubeDataCollector
from src.utils.config import get_data_path, get_youtube_credentials

## Prerequisites

Before running the collection pipeline, ensure the following are in place:

1. **Google Cloud Project** with the YouTube Data API v3 enabled.
2. **OAuth 2.0 credentials** (Desktop application type).
3. **Environment variables** set in a `.env` file at the project root:

```
YOUTUBE_CLIENT_ID=your_client_id_here
YOUTUBE_CLIENT_SECRET=your_client_secret_here
YOUTUBE_PROJECT_ID=your_project_id_here
```

4. **Input data** -- `data/raw/YoutubeNodes.csv` with columns `Id` and `Label`.

The first time you run the collector, a browser window will open for OAuth consent.
After authentication, a `token.pickle` file is saved for subsequent runs.

In [ ]:
# Verify credential environment variables are set (values are not printed)
creds = get_youtube_credentials()
for key, value in creds.items():
    status = 'SET' if value else 'NOT SET'
    print(f"  {key}: {status}")

## Collector Configuration

| Parameter | Description | Default |
|-----------|-------------|--------|
| `input_file` | Path to node list CSV | `data/raw/YoutubeNodes.csv` |
| `output_file` | Path for aggregated output CSV | `youtube_channel_data.csv` |
| `video_output_dir` | Directory for per-channel video CSVs | `video_data/` |
| `max_videos` | Maximum videos to fetch per channel | 200 |
| `quota_limit` | Stop collection after this many API units | 9000 |
| `checkpoint_file` | Path for resumable checkpoint JSON | `youtube_checkpoint.json` |

In [ ]:
# Configuration
INPUT_FILE = str(get_data_path('raw') / 'YoutubeNodes.csv')
OUTPUT_FILE = str(get_data_path('raw') / 'youtube_channel_data.csv')
VIDEO_OUTPUT_DIR = str(get_data_path('raw') / 'video_data')
MAX_VIDEOS = 200
QUOTA_LIMIT = 9000

print(f"Input file:       {INPUT_FILE}")
print(f"Output file:      {OUTPUT_FILE}")
print(f"Video output dir: {VIDEO_OUTPUT_DIR}")
print(f"Max videos/channel: {MAX_VIDEOS}")
print(f"Quota limit:      {QUOTA_LIMIT}")

## Inspect the Input Data

The input file contains YouTube channel IDs and labels extracted from the subscription
network. We verify its structure before running the collector.

In [ ]:
# Load and inspect the input node list
if os.path.exists(INPUT_FILE):
    nodes_df = pd.read_csv(INPUT_FILE)
    print(f"Input file loaded: {len(nodes_df)} channels")
    print(f"Columns: {list(nodes_df.columns)}")
    print()
    print(nodes_df.head())
else:
    print(f"Input file not found at: {INPUT_FILE}")
    print("Please ensure YoutubeNodes.csv is in the data/raw/ directory.")

## Initialize the Collector

The `YouTubeDataCollector` automatically loads any existing checkpoint, so interrupted
collection runs resume where they left off.

In [ ]:
collector = YouTubeDataCollector(
    input_file=INPUT_FILE,
    output_file=OUTPUT_FILE,
    video_output_dir=VIDEO_OUTPUT_DIR,
    max_videos=MAX_VIDEOS,
    quota_limit=QUOTA_LIMIT,
)

print(f"Collector initialized.")
print(f"  Already processed: {len(collector.processed_indices)} channels")
print(f"  Quota used so far: {collector.quota_used}")

## Run Data Collection

**WARNING:** The cell below will make real API calls and consume YouTube API quota.
Only run it when you have valid credentials and intend to collect data.

The collector:
1. Authenticates via OAuth2 (browser popup on first run).
2. Iterates through unprocessed channels.
3. Fetches channel statistics and video metadata.
4. Saves a checkpoint after each channel.
5. Stops when the quota limit is reached or all channels are processed.

You can safely interrupt the process with `Ctrl+C` -- the checkpoint will be saved.

In [ ]:
# Uncomment the lines below to run the actual collection.
# This requires valid YouTube API credentials.

# channel_data = collector.collect_data()
# print(f"\nCollection complete. {len(channel_data)} channels processed.")
# print(f"Total quota used: {collector.quota_used}")

## Inspect Collected Data

After collection, each channel's video data is stored in a separate CSV file within
`video_output_dir`. We can inspect the collected output to verify quality.

In [ ]:
# Check how many video data files exist
if os.path.exists(VIDEO_OUTPUT_DIR):
    video_files = [f for f in os.listdir(VIDEO_OUTPUT_DIR) if f.endswith('_videos.csv')]
    print(f"Video data files found: {len(video_files)}")
    if video_files:
        # Show a sample file
        sample_file = os.path.join(VIDEO_OUTPUT_DIR, video_files[0])
        sample_df = pd.read_csv(sample_file)
        print(f"\nSample: {video_files[0]}")
        print(f"  Videos: {len(sample_df)}")
        print(f"  Columns: {list(sample_df.columns)}")
        print(sample_df.head())
else:
    print(f"Video output directory does not exist yet: {VIDEO_OUTPUT_DIR}")
    print("Run the collection step above to populate it.")